# 🚀 DataMate Agent - Google Colab Training & Evaluation Pipeline
### Zalo AI Challenge: Vietnamese Offline Tabular QA System

Notebook này thiết lập môi trường GPU, tự động tải dữ liệu từ Google Drive, chạy vòng lặp **Iterative Self-Training (QLoRA + Rejection Sampling)** và đánh giá điểm số **Penalized Accuracy (PA)**.

## 1. Kiểm tra Phần cứng GPU (Khuyến nghị T4, A100 hoặc L4)

In [ ]:
!nvidia-smi

## 2. Clone Repository từ GitHub

In [ ]:
# Clone mã nguồn mới nhất từ repo của bạn
!rm -rf /content/datamate_zalo_ai
!git clone https://github.com/tuannh1905/datamate_zalo_ai.git /content/datamate_zalo_ai
%cd /content/datamate_zalo_ai/datamate_agent

## 3. Cài đặt Môi trường (vLLM, DuckDB, Unsloth, HuggingFace Base Weights)

In [ ]:
# Cài đặt dependencies và tải weights offline Qwen2.5-Coder-7B-Instruct
!bash scripts/setup_colab.sh

## 4. Tải Dữ liệu Cuộc thi từ Google Drive Folder
Bạn có thể chọn **Cách 1** (Tải tự động bằng `gdown`) hoặc **Cách 2** (Mount Google Drive).

In [ ]:
# CÁCH 1 (Khuyên dùng): Tải trực tiếp folder qua Google Drive Link
!pip install -q gdown
!mkdir -p /content/dataset_gdrive
!gdown --folder "https://drive.google.com/drive/folders/1sMB8d47li018WyG_SsXIZJ02jfL8OaLi" -O /content/dataset_gdrive

# Kiểm tra dữ liệu đã tải thành công
!ls -la /content/dataset_gdrive

In [ ]:
# CÁCH 2: Mount Drive nếu bạn đã lưu folder vào MyDrive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_BASE_DIR = "/content/drive/MyDrive/Tên_Thư_Mục_Của_Bạn"

## 5. Chạy Thử Nghiệm Nhanh (Smoke Test trên 5 câu hỏi)

In [ ]:
# Chạy thử suy luận và chấm điểm nhanh 5 câu hỏi để xác nhận luồng chạy trơn tru
!python scripts/run_eval.py \
    --dataset train \
    --drive_base_path /content/dataset_gdrive \
    --run_inference \
    --limit 5 \
    --verbose

## 6. Chạy Vòng lặp Tự huấn luyện (Iterative Self-Training Loop)
Quy trình tự học 3 chu kỳ:
- **Vòng 1:** Model Base sinh 12 mẫu $\to$ Rejection Sampling chọn đáp án đúng $\to$ Huấn luyện LoRA Adapter `v1`.
- **Vòng 2:** Model + Adapter `v1` sinh mẫu $\to$ Rejection Sampling $\to$ Huấn luyện LoRA Adapter `v2`.
- **Vòng 3:** Model + Adapter `v2` sinh mẫu $\to$ Rejection Sampling $\to$ Huấn luyện Adapter `v3` hoàn chỉnh.

In [ ]:
!DRIVE_BASE_PATH="/content/dataset_gdrive" \
 DATASET="train" \
 bash scripts/run_self_training.sh

## 7. Đánh giá Toàn diện với Adapter v3 và Đo lường Penalized Accuracy

In [ ]:
!python scripts/run_eval.py \
    --dataset train \
    --drive_base_path /content/dataset_gdrive \
    --adapter ./weights/adapters/v3 \
    --run_inference

## 8. Sao lưu Adapter Weights vào Google Drive cá nhân

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ZaloAI_DataMate_Backup
!cp -r ./weights/adapters /content/drive/MyDrive/ZaloAI_DataMate_Backup/
print("✓ Đã sao lưu adapter weights thành công vào Google Drive!")